In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
app_train = pd.read_csv("../Data/app_train_merged.csv")
app_test = pd.read_csv("../Data/app_test_merged.csv")

### Gộp 2 train/test lại với nhau
## loại cột Target

In [ ]:
# Tách Target
target = app_train["TARGET"]
app_train = app_train.drop(columns=["TARGET"])
n_train = len(app_train) # lưu lại trước

# Gộp 2 train/test lại với nhau
app_all = pd.concat([app_train, app_test], axis=0).reset_index(drop=True)

In [ ]:
app_all.shape

In [ ]:
app_all.columns.to_list()

### Gộp 2 bảng train/test nhưng chưa xử lí cột ngày của test nên giờ xử lí lần nữa trên toàn bộ dataset mới
### Tương tự, một số cột chưa có giá trị ví dụ EXT__MISSING

In [ ]:
day_cols = [
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE",
]

for col in day_cols:
    app_all[col] = app_all[col].abs()

# Tạo cột mới, quét toàn bộ day_employed, nếu có giá trị nào bằng 365243 thì gán 1, ngược lại gán 0
app_all["DAYS_EMPLOYED_ANOM"] = (app_all["DAYS_EMPLOYED"] == 365243).astype(int)

# Thay 365243 bằng giá trị NaN để không ảnh hưởng đến việc chia bin
app_all["DAYS_EMPLOYED"] = app_all["DAYS_EMPLOYED"].replace(365243, np.nan)

# Chuyển ngày thành năm
for col in day_cols:
    app_all[col] = app_all[col] / 365

In [ ]:
# EXT_SOURCE_1_MISSING và EXT_SOURCE_3_MISSING
app_all["EXT_SOURCE_1_MISSING"] = app_all["EXT_SOURCE_1"].isnull().astype(int)
app_all["EXT_SOURCE_3_MISSING"] = app_all["EXT_SOURCE_3"].isnull().astype(int)

# BDS_INFO_MISSING — thiếu bất kỳ cột nào trong nhóm MODE
mode_cols = [
    col
    for col in app_all.columns
    if col.endswith("_MODE")
    and col
    not in [
        "FONDKAPREMONT_MODE",
        "HOUSETYPE_MODE",
        "WALLSMATERIAL_MODE",
        "EMERGENCYSTATE_MODE",
    ]
]
app_all["BDS_INFO_MISSING"] = app_all[mode_cols].isnull().any(axis=1).astype(int)

# AMT_REQ_MISSING — thiếu toàn bộ cột AMT_REQ
amt_req_cols = [
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]
app_all["AMT_REQ_MISSING"] = app_all[amt_req_cols].isnull().any(axis=1).astype(int)

# OWN_CAR_AGE_MISSING
app_all["OWN_CAR_AGE_MISSING"] = app_all["OWN_CAR_AGE"].isnull().astype(int)

# Danh sách và ý nghĩa các cột trong app_train_merged

## 1. Thông tin định danh
| Cột | Ý nghĩa |
|---|---|
| `SK_ID_CURR` | ID khách hàng |

---

## 2. Thông tin khoản vay hiện tại
| Cột | Ý nghĩa |
|---|---|
| `NAME_CONTRACT_TYPE` | Loại hợp đồng (Cash loans / Revolving loans) |
| `AMT_CREDIT` | Số tiền vay |
| `AMT_ANNUITY` | Khoản trả góp hàng tháng |
| `AMT_GOODS_PRICE` | Giá trị hàng hóa được mua |
| `AMT_INCOME_TOTAL` | Thu nhập hàng năm của khách hàng |

---

## 3. Thông tin cá nhân
| Cột | Ý nghĩa |
|---|---|
| `CODE_GENDER` | Giới tính (F/M/XNA) |
| `FLAG_OWN_CAR` | Có xe không (Y/N) |
| `FLAG_OWN_REALTY` | Có bất động sản không (Y/N) |
| `CNT_CHILDREN` | Số con |
| `CNT_FAM_MEMBERS` | Số thành viên gia đình |
| `NAME_TYPE_SUITE` | Người đi cùng khi nộp đơn |
| `NAME_INCOME_TYPE` | Loại thu nhập (Working/Pensioner/...) |
| `NAME_EDUCATION_TYPE` | Trình độ học vấn |
| `NAME_FAMILY_STATUS` | Tình trạng hôn nhân |
| `NAME_HOUSING_TYPE` | Loại chỗ ở |
| `OCCUPATION_TYPE` | Nghề nghiệp |
| `DAYS_BIRTH` | Tuổi (số ngày, đã convert dương) |
| `DAYS_EMPLOYED` | Số ngày đã làm việc tại công ty hiện tại |
| `DAYS_REGISTRATION` | Số ngày kể từ khi đăng ký giấy tờ |
| `DAYS_ID_PUBLISH` | Số ngày kể từ khi cấp CMND |
| `DAYS_LAST_PHONE_CHANGE` | Số ngày kể từ lần cuối đổi SĐT |
| `OWN_CAR_AGE` | Tuổi xe (năm) |
| `ORGANIZATION_TYPE` | Loại tổ chức làm việc |

---

## 4. Thông tin khu vực
| Cột | Ý nghĩa |
|---|---|
| `REGION_POPULATION_RELATIVE` | Mật độ dân số khu vực |
| `REGION_RATING_CLIENT` | Xếp hạng khu vực (1-3) |
| `REGION_RATING_CLIENT_W_CITY` | Xếp hạng khu vực có tính thành phố (1-3) |
| `REG_REGION_NOT_LIVE_REGION` | Địa chỉ đăng ký ≠ địa chỉ ở (cấp vùng) |
| `REG_REGION_NOT_WORK_REGION` | Địa chỉ đăng ký ≠ địa chỉ làm việc (cấp vùng) |
| `LIVE_REGION_NOT_WORK_REGION` | Địa chỉ ở ≠ địa chỉ làm việc (cấp vùng) |
| `REG_CITY_NOT_LIVE_CITY` | Địa chỉ đăng ký ≠ địa chỉ ở (cấp thành phố) |
| `REG_CITY_NOT_WORK_CITY` | Địa chỉ đăng ký ≠ địa chỉ làm việc (cấp thành phố) |
| `LIVE_CITY_NOT_WORK_CITY` | Địa chỉ ở ≠ địa chỉ làm việc (cấp thành phố) |
| `WEEKDAY_APPR_PROCESS_START` | Ngày trong tuần nộp đơn |
| `HOUR_APPR_PROCESS_START` | Giờ nộp đơn |

---

## 5. Thông tin bất động sản (_AVG, _MODE, _MEDI)
| Cột | Ý nghĩa |
|---|---|
| `APARTMENTS_AVG/MODE/MEDI` | Diện tích căn hộ (chuẩn hóa 0-1) |
| `BASEMENTAREA_AVG/MODE/MEDI` | Diện tích tầng hầm |
| `YEARS_BEGINEXPLUATATION_AVG/MODE/MEDI` | Năm bắt đầu khai thác tòa nhà |
| `YEARS_BUILD_AVG/MODE/MEDI` | Năm xây dựng tòa nhà |
| `COMMONAREA_AVG/MODE/MEDI` | Diện tích khu vực chung |
| `ELEVATORS_AVG/MODE/MEDI` | Số thang máy |
| `ENTRANCES_AVG/MODE/MEDI` | Số lối vào |
| `FLOORSMAX_AVG/MODE/MEDI` | Số tầng tối đa |
| `FLOORSMIN_AVG/MODE/MEDI` | Số tầng tối thiểu |
| `LANDAREA_AVG/MODE/MEDI` | Diện tích đất |
| `LIVINGAPARTMENTS_AVG/MODE/MEDI` | Số căn hộ đang ở |
| `LIVINGAREA_AVG/MODE/MEDI` | Diện tích sống |
| `NONLIVINGAPARTMENTS_AVG/MODE/MEDI` | Số căn hộ không ở |
| `NONLIVINGAREA_AVG/MODE/MEDI` | Diện tích không ở |
| `TOTALAREA_MODE` | Tổng diện tích (chỉ có _MODE) |
| `FONDKAPREMONT_MODE` | Loại quỹ sửa chữa |
| `HOUSETYPE_MODE` | Loại nhà |
| `WALLSMATERIAL_MODE` | Vật liệu tường |
| `EMERGENCYSTATE_MODE` | Trạng thái khẩn cấp của tòa nhà |

---

## 6. Điểm tín dụng bên ngoài
| Cột | Ý nghĩa |
|---|---|
| `EXT_SOURCE_1` | Điểm tín dụng từ nguồn bên ngoài 1 (0-1, càng cao càng tốt) |
| `EXT_SOURCE_2` | Điểm tín dụng từ nguồn bên ngoài 2 |
| `EXT_SOURCE_3` | Điểm tín dụng từ nguồn bên ngoài 3 |

---

## 7. Thông tin liên lạc (FLAG)
| Cột | Ý nghĩa |
|---|---|
| `FLAG_MOBIL` | Có cung cấp số di động không |
| `FLAG_EMP_PHONE` | Có cung cấp SĐT công ty không |
| `FLAG_WORK_PHONE` | Có cung cấp SĐT nơi làm việc không |
| `FLAG_CONT_MOBILE` | Số di động có liên lạc được không |
| `FLAG_PHONE` | Có cung cấp số điện thoại không |
| `FLAG_EMAIL` | Có cung cấp email không |

---

## 8. Giấy tờ (FLAG_DOCUMENT)
| Cột | Ý nghĩa |
|---|---|
| `FLAG_DOCUMENT_2` đến `FLAG_DOCUMENT_21` | Khách hàng có nộp giấy tờ loại X không (0/1) |

---

## 9. Vòng tròn xã hội
| Cột | Ý nghĩa |
|---|---|
| `OBS_30_CNT_SOCIAL_CIRCLE` | Số người trong vòng tròn xã hội được quan sát trong 30 ngày |
| `DEF_30_CNT_SOCIAL_CIRCLE` | Số người trong vòng tròn xã hội vỡ nợ trong 30 ngày |
| `OBS_60_CNT_SOCIAL_CIRCLE` | Số người trong vòng tròn xã hội được quan sát trong 60 ngày |
| `DEF_60_CNT_SOCIAL_CIRCLE` | Số người trong vòng tròn xã hội vỡ nợ trong 60 ngày |

---

## 10. Tra cứu tín dụng (AMT_REQ_CREDIT_BUREAU)
| Cột | Ý nghĩa |
|---|---|
| `AMT_REQ_CREDIT_BUREAU_HOUR` | Số lần tra cứu tín dụng trong 1 giờ trước khi nộp đơn |
| `AMT_REQ_CREDIT_BUREAU_DAY` | Số lần tra cứu trong 1 ngày |
| `AMT_REQ_CREDIT_BUREAU_WEEK` | Số lần tra cứu trong 1 tuần |
| `AMT_REQ_CREDIT_BUREAU_MON` | Số lần tra cứu trong 1 tháng |
| `AMT_REQ_CREDIT_BUREAU_QRT` | Số lần tra cứu trong 1 quý |
| `AMT_REQ_CREDIT_BUREAU_YEAR` | Số lần tra cứu trong 1 năm |

---

## 11. Flags đặc biệt (tạo thêm trong quá trình xử lý)
| Cột | Ý nghĩa |
|---|---|
| `DAYS_EMPLOYED_ANOM` | 1 = không đi làm chính thức (hưu trí/nội trợ), 0 = bình thường |
| `EXT_SOURCE_1_MISSING` | 1 = không có điểm tín dụng nguồn 1 |
| `EXT_SOURCE_3_MISSING` | 1 = không có điểm tín dụng nguồn 3 |
| `BDS_INFO_MISSING` | 1 = không có thông tin bất động sản |
| `AMT_REQ_MISSING` | 1 = không có lịch sử tra cứu tín dụng |
| `OWN_CAR_AGE_MISSING` | 1 = không có xe |
| `card_missing` | 1 = không có thẻ tín dụng |

---

## 12. Thông tin từ bureau (lịch sử tín dụng tại Credit Bureau)

### 12a. Số tiền
| Cột | Ý nghĩa |
|---|---|
| `bureau_amt_credit_sum_sum` | Tổng giá trị tất cả khoản vay bureau |
| `bureau_amt_credit_sum_mean` | Trung bình giá trị khoản vay bureau |
| `bureau_amt_credit_debt_sum` | Tổng dư nợ hiện tại tại bureau |
| `bureau_amt_credit_debt_mean` | Trung bình dư nợ hiện tại |
| `bureau_amt_credit_limit_sum` | Tổng hạn mức tín dụng (thẻ) tại bureau |
| `bureau_amt_credit_limit_mean` | Trung bình hạn mức tín dụng |
| `bureau_amt_annuity_sum` | Tổng khoản trả góp tại bureau |
| `bureau_amt_annuity_mean` | Trung bình khoản trả góp |

### 12b. Trạng thái khoản vay
| Cột | Ý nghĩa |
|---|---|
| `bureau_active_rate_active` | Tỷ lệ % khoản vay đang hoạt động |
| `bureau_active_rate_bad debt` | Tỷ lệ % khoản vay nợ xấu |
| `bureau_active_rate_closed` | Tỷ lệ % khoản vay đã đóng |
| `bureau_active_rate_sold` | Tỷ lệ % khoản vay đã bán nợ |
| `bureau_count` | Tổng số khoản vay tại bureau |

### 12c. Loại khoản vay
| Cột | Ý nghĩa |
|---|---|
| `bureau_type_rate_car_loan` | Tỷ lệ % khoản vay mua xe |
| `bureau_type_rate_consumer_credit` | Tỷ lệ % khoản vay tiêu dùng |
| `bureau_type_rate_credit_card` | Tỷ lệ % thẻ tín dụng |
| `bureau_type_rate_microloan` | Tỷ lệ % vay vi mô |
| `bureau_type_rate_mortgage` | Tỷ lệ % vay thế chấp |
| `bureau_type_rate_other` | Tỷ lệ % loại khác |

### 12d. Thời gian
| Cột | Ý nghĩa |
|---|---|
| `bureau_day_credit_min` | Khoản vay gần nhất mở cách đây bao nhiêu ngày |
| `bureau_day_credit_max` | Khoản vay đầu tiên mở cách đây bao nhiêu ngày |
| `bureau_day_credit_mean` | Trung bình số ngày kể từ khi mở khoản vay |
| `bureau_days_credit_enddate_mean` | Trung bình thời hạn còn lại |
| `bureau_day_enddate_fact_min` | Khoản vay đóng gần nhất cách đây bao nhiêu ngày |
| `bureau_day_credit_update_min` | Lần cập nhật thông tin gần nhất |

### 12e. Quá hạn hiện tại
| Cột | Ý nghĩa |
|---|---|
| `bureau_credit_day_overdue_max` | Số ngày quá hạn cao nhất hiện tại |
| `bureau_amt_credit_max_overdue` | Số tiền quá hạn cao nhất từng có (lịch sử) |
| `bureau_amt_credit_sum_overdue_sum` | Tổng số tiền đang quá hạn hiện tại |
| `bureau_amt_credit_sum_overdue_max` | Khoản quá hạn lớn nhất hiện tại |
| `bureau_cnt_credit_prolong_sum` | Tổng số lần gia hạn khoản vay |

### 12f. DPD từ bureau_balance
| Cột | Ý nghĩa |
|---|---|
| `bureau_dpd_max` | Mức DPD cao nhất từng đạt (across tất cả khoản vay) |
| `bureau_dpd_ever_rate` | Tỷ lệ % khoản vay từng quá hạn |
| `bureau_dpd_severe_ever_rate` | Tỷ lệ % khoản vay từng quá hạn nặng (status >= 3) |
| `bureau_dpd_ratio_mean` | Trung bình tỷ lệ tháng quá hạn (%) |
| `bureau_months_since_last_dpd_min` | Quá hạn gần nhất cách đây bao nhiêu tháng |
| `bureau_balance_missing_rate` | Tỷ lệ % khoản vay thiếu lịch sử balance |
| `bureau_months_count_sum` | Tổng số tháng lịch sử tín dụng |
| `bureau_dpd_count_sum` | Tổng số tháng từng quá hạn |
| `bureau_active_months_sum` | Tổng số tháng có dữ liệu hoạt động |

### 12g. Số tháng theo từng trạng thái
| Cột | Ý nghĩa |
|---|---|
| `bureau_status_count_0_sum` | Tổng tháng không quá hạn |
| `bureau_status_count_1_sum` | Tổng tháng quá hạn 1-30 ngày |
| `bureau_status_count_2_sum` | Tổng tháng quá hạn 31-60 ngày |
| `bureau_status_count_3_sum` | Tổng tháng quá hạn 61-90 ngày |
| `bureau_status_count_4_sum` | Tổng tháng quá hạn 91-120 ngày |
| `bureau_status_count_5_sum` | Tổng tháng quá hạn >120 ngày hoặc nợ xấu |
| `bureau_status_count_C_sum` | Tổng tháng khoản vay đã đóng |
| `bureau_status_count_X_sum` | Tổng tháng không có thông tin |

### 12h. Tỷ lệ trạng thái tháng gần nhất
| Cột | Ý nghĩa |
|---|---|
| `bureau_status_last_rate_0` | Tỷ lệ % khoản vay có tháng gần nhất = không quá hạn |
| `bureau_status_last_rate_1` | Tỷ lệ % khoản vay có tháng gần nhất = quá hạn 1-30 ngày |
| `bureau_status_last_rate_2` | Tỷ lệ % khoản vay có tháng gần nhất = quá hạn 31-60 ngày |
| `bureau_status_last_rate_3` | Tỷ lệ % khoản vay có tháng gần nhất = quá hạn 61-90 ngày |
| `bureau_status_last_rate_4` | Tỷ lệ % khoản vay có tháng gần nhất = quá hạn 91-120 ngày |
| `bureau_status_last_rate_5` | Tỷ lệ % khoản vay có tháng gần nhất = quá hạn >120 ngày |
| `bureau_status_last_rate_C` | Tỷ lệ % khoản vay đã đóng ở tháng gần nhất |
| `bureau_status_last_rate_X` | Tỷ lệ % khoản vay không có thông tin ở tháng gần nhất |

---

## 13. Thông tin từ previous_application (đơn vay trước tại Home Credit)
| Cột | Ý nghĩa |
|---|---|
| `pre_loan_count` | Tổng số đơn vay trước đây |
| `pre_approved_rate` | Tỷ lệ % đơn được duyệt |
| `pre_refused_rate` | Tỷ lệ % đơn bị từ chối |
| `pre_amt_credit_sum` | Tổng số tiền vay được duyệt trước đây |
| `pre_amt_application_sum` | Tổng số tiền xin vay trước đây |
| `pre_amt_annuity_sum` | Tổng khoản trả góp các lần vay trước |
| `pre_amt_goodsprice_sum` | Tổng giá trị hàng hóa các lần vay trước |
| `pre_amt_credit_mean` | Trung bình số tiền vay được duyệt |
| `pre_amt_application_mean` | Trung bình số tiền xin vay |
| `pre_amt_annuity_mean` | Trung bình khoản trả góp |
| `pre_amt_goodsprice_mean` | Trung bình giá trị hàng hóa |

---

## 14. Thông tin từ credit_card_balance (lịch sử thẻ tín dụng tại Home Credit)
| Cột | Ý nghĩa |
|---|---|
| `card_months_count` | Số tháng có lịch sử thẻ tín dụng |
| `card_amt_balance_mean` | Số dư trung bình hàng tháng |
| `card_amt_balance_max` | Số dư cao nhất từng có |
| `card_amt_credit_limit_mean` | Hạn mức tín dụng trung bình |
| `card_amt_drawings_mean` | Số tiền giao dịch trung bình hàng tháng |
| `card_amt_payment_mean` | Số tiền thanh toán trung bình hàng tháng |
| `card_amt_payment_total_mean` | Tổng tiền thanh toán trung bình (bao gồm phí, lãi) |
| `card_dpd_max` | Số ngày quá hạn cao nhất |
| `card_dpd_mean` | Số ngày quá hạn trung bình |
| `card_dpd_def_max` | Số ngày quá hạn (có tolerance) cao nhất |
| `card_dpd_ever_count` | Số tháng từng quá hạn |
| `card_utilization_mean` | Tỷ lệ sử dụng hạn mức trung bình |
| `card_utilization_max` | Tỷ lệ sử dụng hạn mức cao nhất |

---

## 15. Thông tin từ POS_CASH_balance (lịch sử khoản vay POS/Cash)
| Cột | Ý nghĩa |
|---|---|
| `pos_months_count` | Số tháng có lịch sử POS/Cash |
| `pos_dpd_max` | Số ngày quá hạn cao nhất |
| `pos_dpd_mean` | Số ngày quá hạn trung bình |
| `pos_dpd_def_max` | Số ngày quá hạn (có tolerance) cao nhất |
| `pos_dpd_ever_count` | Số tháng từng quá hạn |
| `pos_cnt_instalment_mean` | Trung bình tổng số kỳ trả |
| `pos_cnt_instalment_future_mean` | Trung bình số kỳ còn lại phải trả |
| `pos_high_risk_ever` | Từng có trạng thái High_risk (Demand/Amortized debt) chưa |
| `pos_high_risk_rate` | Tỷ lệ % tháng ở trạng thái High_risk |
| `pos_completed_rate` | Tỷ lệ % tháng ở trạng thái Completed |

---

## 16. Thông tin từ installments_payments (lịch sử từng kỳ thanh toán)
| Cột | Ý nghĩa |
|---|---|
| `install_count` | Tổng số kỳ thanh toán đã ghi nhận |
| `install_payment_delay_mean` | Trung bình số ngày trễ/sớm (dương=trễ, âm=sớm) |
| `install_payment_delay_max` | Số ngày trễ cao nhất từng có |
| `install_late_count` | Số kỳ từng trả trễ |
| `install_late_rate` | Tỷ lệ % kỳ trả trễ |
| `install_payment_ratio_mean` | Trung bình tỷ lệ thực trả/phải trả (>=1 tốt, <1 xấu) |
| `install_payment_ratio_min` | Tỷ lệ thực trả/phải trả thấp nhất (bắt kỳ trả thiếu nhất) |
| `install_payment_diff_mean` | Trung bình số tiền thiếu/dư mỗi kỳ |
| `install_payment_diff_sum` | Tổng số tiền thiếu/dư cộng dồn |
| `install_version_max` | Số lần thay đổi lịch trả nợ cao nhất (gia hạn/tái cơ cấu) |
| `install_credit_card_rate` | Tỷ lệ % kỳ trả là thẻ tín dụng (version=0) |

# Danh sách Feature Engineering Cuối Cùng (Đã cập nhật)

## Nguyên tắc chung
- **Giữ tất cả feature gốc** — không xóa khi dùng tree-based model (XGBoost/LightGBM)
- **Chỉ xóa khi dùng Logistic Regression** — loại bỏ feature tương quan cao
- **Không tạo quá nhiều feature rác** — sau khi tạo xong, dùng feature importance để lọc
- **KHÔNG impute đè lên cột gốc** — LightGBM/XGBoost tự xử lý NaN nội tại rất tốt. Chỉ dùng `skipna=True` hoặc `np.nanmean/np.nansum` khi tính toán tổng hợp để tránh NaN lan truyền ra cột mới. Impute đè lên cột gốc sẽ phá hủy tín hiệu "khách hàng thiếu thông tin = rủi ro cao"

---

## Nhóm 1 — EXT_SOURCE (Ưu tiên cao nhất)
| Feature | Công thức | Ghi chú |
|---|---|---|
| `EXT_SOURCE_MEAN` | `mean(EXT_1, EXT_2, EXT_3, skipna=True)` | Dùng `mean(skipna=True)` tránh NaN lan truyền |
| `EXT_SOURCE_STD` | `std(EXT_1, EXT_2, EXT_3, skipna=True)` | Đo độ bất ổn giữa 3 nguồn |
| `EXT_SOURCE_MIN` | `min(EXT_1, EXT_2, EXT_3, skipna=True)` | Quan điểm bi quan nhất |
| `EXT_SOURCE_MAX` | `max(EXT_1, EXT_2, EXT_3, skipna=True)` | Quan điểm lạc quan nhất |
| `EXT_SOURCE_PRODUCT` | `prod(EXT_1, EXT_2, EXT_3, skipna=True)` | Điểm thấp 1 nguồn kéo toàn bộ xuống |
| `EXT_SOURCE_MISSING_COUNT` | `EXT_SOURCE_1_MISSING + EXT_SOURCE_3_MISSING` | Càng thiếu càng rủi ro |
| `EXT_SOURCE_REGION` | `EXT_SOURCE_MEAN * REGION_RATING_CLIENT` | Tương tác vùng miền |

> ⚠️ **Lưu ý:** `EXT_SOURCE_MEAN` và `EXT_SOURCE_PRODUCT` có thể tương quan cao (>0.95). Cứ tạo cả 2, sau khi chạy feature importance thì loại cái yếu hơn.

---

## Nhóm 2 — Sức khỏe tài chính (Tỷ lệ nợ)
| Feature | Công thức | Ghi chú |
|---|---|---|
| `DEBT_TO_INCOME` | `AMT_CREDIT / AMT_INCOME_TOTAL` | Gánh nặng nợ tổng |
| `ANNUITY_TO_INCOME` | `AMT_ANNUITY / AMT_INCOME_TOTAL` | Gánh nặng trả góp hàng tháng |
| `CREDIT_TO_ANNUITY_MONTHS` | `AMT_CREDIT / AMT_ANNUITY` | Số tháng dự kiến trả nợ |
| `IMPLICIT_INTEREST_RATE` | `AMT_ANNUITY / AMT_CREDIT` | Proxy lãi suất ngầm — càng cao = lãi suất cao hoặc kỳ hạn ngắn = rủi ro cao |
| `LTV` | `AMT_CREDIT / AMT_GOODS_PRICE` | Vay nhiều hơn hay ít hơn giá trị hàng |
| `INCOME_PER_CAPITA` | `AMT_INCOME_TOTAL / CNT_FAM_MEMBERS` | Thu nhập khả dụng thực tế mỗi người |
| `TOTAL_ANNUITY` | `AMT_ANNUITY + bureau_amt_annuity_mean` | Tổng gánh nặng trả góp — dùng `mean` để đồng đơn vị hàng tháng |
| `TOTAL_DEBT_TO_INCOME` | `(AMT_CREDIT + bureau_amt_credit_debt_sum) / AMT_INCOME_TOTAL` | Tổng nợ hiện tại + nợ bureau / thu nhập |
| `BUREAU_DEBT_RATIO` | `bureau_amt_credit_debt_sum / AMT_INCOME_TOTAL` | Nợ bên ngoài so với thu nhập |
| `BUREAU_CREDIT_UTILIZATION` | `bureau_amt_credit_debt_sum / bureau_amt_credit_sum_sum` | Tỷ lệ sử dụng hạn mức tại bureau |

---

## Nhóm 3 — Hành vi thanh toán tổng hợp (Cross-source DPD)
| Feature | Công thức | Ghi chú |
|---|---|---|
| `MAX_DPD_ALL_SOURCES` | `max(bureau_dpd_max, card_dpd_max, pos_dpd_max, install_payment_delay_max)` | DPD nặng nhất từ mọi nguồn |
| `TOTAL_LATE_MONTHS` | `bureau_dpd_count_sum + card_dpd_ever_count + pos_dpd_ever_count + install_late_count` | Tổng số lần/tháng từng quá hạn |
| `HAS_SEVERE_OVERDUE` | `1` nếu `bureau_dpd_severe_ever_rate > 0` hoặc `pos_high_risk_ever = 1` | Từng quá hạn nghiêm trọng |
| `DPD_RECENCY_SEVERITY` | `bureau_dpd_max / (abs(bureau_months_since_last_dpd_min) + 1)` | Quá hạn nặng + gần đây = rủi ro cao |
| `INSTALL_COMPLETION_RATE` | Xem ghi chú bên dưới | Tỷ lệ kỳ trả đúng hạn tại Home Credit |
| `INSTALLMENT_PAYMENT_GAP` | `install_payment_ratio_mean - 1.0` | Âm = thường xuyên trả thiếu |

> ⚠️ **`INSTALL_COMPLETION_RATE` — 2 phiên bản:**
> - **Phiên bản A (để model tự xử lý):** `(install_count - install_late_count) / install_count` → nếu `install_count = 0` hoặc `NaN` thì giữ nguyên `NaN`, để LightGBM/XGBoost tự xử lý
> - **Phiên bản B (coi thiếu = rủi ro):** Tương tự nhưng nếu `install_count = 0` hoặc `NaN` thì đặt = `0` — ngụ ý "không có lịch sử tốt = rủi ro"
>
> Nên tạo cả 2 cột (`INSTALL_COMPLETION_RATE_NAN` và `INSTALL_COMPLETION_RATE_ZERO`), chạy feature importance để xem cái nào tốt hơn.

---

## Nhóm 4 — So sánh với lịch sử vay (Cross-table)
| Feature | Công thức | Ghi chú |
|---|---|---|
| `CURRENT_VS_PREV_CREDIT` | `AMT_CREDIT / pre_amt_credit_mean` | Vay mới lớn hơn vay cũ bao nhiêu lần |
| `PRE_CREDIT_GAP` | `pre_amt_credit_mean - pre_amt_application_mean` | Âm = thường được duyệt ít hơn xin |
| `PRE_APPROVAL_RATE` | `pre_approved_rate / (pre_approved_rate + pre_refused_rate)` | Tỷ lệ được duyệt trong số đã quyết định (loại trừ Canceled/Unused); thay thành Nan nếu khách chưa được duyệt/từ chối lần nào |
| `PRE_CREDIT_TO_GOODS` | `pre_amt_credit_mean / pre_amt_goodsprice_mean` | Tỷ lệ vay/giá hàng ở lần vay trước |

---

## Nhóm 5 — Độ ổn định nhân thân
| Feature | Công thức | Ghi chú |
|---|---|---|
| `EMPLOYMENT_LIFE_RATIO` | `DAYS_EMPLOYED / DAYS_BIRTH` | % cuộc đời đã đi làm — NaN nếu không đi làm (đã xử lý DAYS_EMPLOYED_ANOM trước đó) |
| `RESIDENCE_STABILITY` | `DAYS_REGISTRATION / DAYS_BIRTH` | Ổn định chỗ ở so với tuổi |
| `PHONE_STABILITY` | `DAYS_LAST_PHONE_CHANGE / DAYS_BIRTH` | Ổn định SĐT so với tuổi |
| `ID_STABILITY` | `DAYS_ID_PUBLISH / DAYS_BIRTH` | CMND lâu chưa đổi so với tuổi |
| `ADDRESS_MISMATCH_SCORE` | `REG_REGION_NOT_LIVE_REGION + REG_CITY_NOT_LIVE_CITY + REG_CITY_NOT_WORK_CITY` | Tổng điểm sai lệch địa chỉ |

---

## Nhóm 6 — Thẻ tín dụng (Credit Card Behavior)
| Feature | Công thức | Ghi chú |
|---|---|---|
| `CARD_PAYMENT_TO_BALANCE` | `card_amt_payment_mean / card_amt_balance_mean` | Trả tối thiểu hay trả hết? |
| `CARD_DRAWINGS_TO_LIMIT` | `card_amt_drawings_mean / card_amt_credit_limit_mean` | Tỷ lệ rút tiền so với hạn mức |

---

## Nhóm 7 — Vòng tròn xã hội
| Feature | Công thức | Ghi chú |
|---|---|---|
| `DEF_RATE_30` | `DEF_30_CNT_SOCIAL_CIRCLE / (OBS_30_CNT_SOCIAL_CIRCLE + 1e-9)` | Tỷ lệ vỡ nợ bạn bè 30 ngày |
| `DEF_RATE_60` | `DEF_60_CNT_SOCIAL_CIRCLE / (OBS_60_CNT_SOCIAL_CIRCLE + 1e-9)` | Tỷ lệ vỡ nợ bạn bè 60 ngày |
| `SOCIAL_TREND_WORSE` | `DEF_RATE_60 - DEF_RATE_30` | Dương = bạn bè gần đây vỡ nợ nhiều hơn |

---

## Nhóm 8 — Cờ báo động đỏ (Red Flags)
| Feature | Công thức | Ghi chú |
|---|---|---|
| `RECENT_INQUIRY_SPIKE` | Xem ghi chú bên dưới | Phát hiện cơn khát tín dụng bất thường |
| `TOTAL_MISSING_FLAGS` | `EXT_SOURCE_1_MISSING + EXT_SOURCE_3_MISSING + BDS_INFO_MISSING + AMT_REQ_MISSING + OWN_CAR_AGE_MISSING + card_missing` | Càng nhiều missing càng rủi ro |
| `IS_OFFICIAL_EMPLOYED` | `1` nếu `NAME_INCOME_TYPE` thuộc `['Working', 'State servant']` | Thu nhập chính thức ổn định hơn |

> ⚠️ **`RECENT_INQUIRY_SPIKE` — Logic xử lý:**
> ```
> if YEAR == 0 and WEEK == 0 → = 1.0  (bình thường, không có tra cứu nào)
> if YEAR == 0 and WEEK > 0  → = 2.0  (báo động đỏ, bỗng dưng có tra cứu)
> else                        → = WEEK / (YEAR / 52)  (tỷ lệ tra cứu tuần này vs trung bình)
> ```

---

## Thứ tự thực hiện
1. **Nhóm 1** — EXT_SOURCE (feature mạnh nhất, làm trước)
2. **Nhóm 2** — Tỷ lệ tài chính cốt lõi
3. **Nhóm 3** — Tổng hợp DPD từ 4 nguồn
4. **Nhóm 4** — So sánh với lịch sử vay
5. **Nhóm 5** — Độ ổn định nhân thân
6. **Nhóm 6** — Thẻ tín dụng
7. **Nhóm 7** — Vòng tròn xã hội
8. **Nhóm 8** — Cờ báo động đỏ

---

## Sau khi tạo feature
1. Chạy LightGBM baseline
2. Dùng `feature_importance_` để xem cột nào có giá trị
3. Loại bỏ feature có importance = 0 hoặc quá thấp
4. Kiểm tra correlation giữa các feature mới — loại bỏ nếu correlation > 0.95
5. **Đặc biệt kiểm tra** cặp `EXT_SOURCE_MEAN` vs `EXT_SOURCE_PRODUCT` và `DEBT_TO_INCOME` vs `TOTAL_DEBT_TO_INCOME`

In [ ]:
# NHÓM 1: EXT_SOURCE FEATURES

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

# EXT_SOURCE_MEAN: Trung bình 3 nguồn điểm tín dụng
# skipna=True để tránh NaN lan truyền khi 1-2 nguồn bị thiếu
app_all["EXT_SOURCE_MEAN"] = app_all[ext_cols].mean(axis=1, skipna=True)

# EXT_SOURCE_STD: Độ lệch chuẩn — đo độ bất ổn giữa 3 nguồn
# Khách hàng có STD cao = các nguồn đánh giá mâu thuẫn nhau
app_all["EXT_SOURCE_STD"] = app_all[ext_cols].std(axis=1, skipna=True)

# EXT_SOURCE_MIN: Quan điểm bi quan nhất trong 3 nguồn
app_all["EXT_SOURCE_MIN"] = app_all[ext_cols].min(axis=1, skipna=True)

# EXT_SOURCE_MAX: Quan điểm lạc quan nhất trong 3 nguồn
app_all["EXT_SOURCE_MAX"] = app_all[ext_cols].max(axis=1, skipna=True)

# EXT_SOURCE_PRODUCT: Tích 3 nguồn
# Nếu 1 nguồn thấp sẽ kéo toàn bộ tích xuống
# skipna=True: bỏ qua NaN thay vì trả về NaN
app_all["EXT_SOURCE_PRODCUCT"] = app_all[ext_cols].prod(axis=1, skipna=True)

# EXT_SOURCE_MISSING_COUNT: Số nguồn bị thiếu
# EXT_SOURCE_2 gần như không thiếu (~0.2%) nên chỉ đếm 1 và 3
# Càng nhiều nguồn thiếu -> càng ít thông tin tín dụng -> rủi ro cao hơn
app_all["EXT_SOURCE_MISSING_COUNT"] = (
    app_all["EXT_SOURCE_1_MISSING"] + app_all["EXT_SOURCE_3_MISSING"]
)

# EXT_SOURCE_REGION: Tương tác giữa điểm tín dụng và xếp hạng khu vực
# REGION_RATING_CLIENT: 1 (tốt) -> 3 (xấu)
# EXT_SOURCE_MEAN cao + REGION_RATING thấp = tốt nhất
# EXT_SOURCE_MEAN thấp + REGION_RATING cao = xấu nhất
app_all["EXT_SOURCE_REGION"] = (
    app_all["EXT_SOURCE_MEAN"] * app_all["REGION_RATING_CLIENT"]
)

# Kiểm tra kết quả:
print("Các feature EXT vừa tạo: ")
new_ext_cols = [
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_STD",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_PRODCUCT",
    "EXT_SOURCE_MISSING_COUNT",
    "EXT_SOURCE_REGION",
]
print(app_all[new_ext_cols].describe().round(4))
print("Số mising values: ")
print(app_all[new_ext_cols].isnull().sum())

### Vốn định kiểm tra tương quan theo từng nhóm và loại bỏ nếu cần nhưng dùng treebase model và feature importance là đủ

In [ ]:
# Nhóm 2: Sức khỏe tài chính

# DEBT_TO_INCOME: Gánh nặng nợ tổng so với thu nhập
# Càng cao = vay quá nhiều so với khả năng
app_all["DEBT_TO_INCOME"] = app_all["AMT_CREDIT"] / app_all["AMT_INCOME_TOTAL"]

# ANNUITY_TO_INCOME: Gánh nặng trả góp hàng tháng so với thu nhập
# Thực tế tổ chức tín dụng thường giới hạn chỉ số này < 40-50%
# (AI chế ra cái này, chưa xem tài liệu kiểm chứng)
app_all["ANNUITY_TO_INCOME"] = app_all["AMT_ANNUITY"] / app_all["AMT_INCOME_TOTAL"]

# CREDIT_TO_ANNUITY_MONTHS: Số tháng dự kiến trả hết nợ
# Kỳ hạn dài = gánh nặng kéo dài (càng lớn càng rủi ro)
app_all["CREDIT_TO_ANNUITY_MONTHS"] = app_all["AMT_CREDIT"] / app_all["AMT_ANNUITY"]

# IMPLICIT_INTEREST_RATE: Proxy lãi suất ngầm
# Càng cao = lãi suất cao hoặc kỳ hạn ngắn
# Nghịch đảo của CREDIT_TO_ANNUITY_MONTHS nhưng mang ý nghĩa khác
app_all["IMPLICIT_INTEREST_RATE"] = (
    app_all["AMT_ANNUITY"] / app_all["AMT_CREDIT"]
).round(4)

# LTV (Loan-to-Value): Tỷ lệ vay so với giá trị hàng hóa
# > 1 = vay nhiều hơn giá trị hàng (có thể vay thêm phí, bảo hiểm...)
app_all["LTV"] = app_all["AMT_CREDIT"] / app_all["AMT_GOODS_PRICE"]

# INCOME_PER_CAPITA: Thu nhập khả dụng thực tế mỗi người trong gia đình
# Gia đình đông người = thu nhập bị chia nhỏ hơn
app_all["INCOME_PER_CAPITA"] = app_all["AMT_INCOME_TOTAL"] / app_all["CNT_FAM_MEMBERS"]

# TOTAL_ANNUITY: Tổng gánh nặng trả góp (khoản vay hiện tại + trung bình các khoản vay bureau)
# Dùng mean để đồng đơn vị hàng tháng với AMT_ANNUITY
app_all["TOTAL_ANNUITY"] = app_all["AMT_ANNUITY"] + app_all[
    "bureau_amt_annuity_mean"
].fillna(
    0
)  # đề phòng không có lịch sử bureau

# Xử lí các outlier âm tại đây
bureau_debt_clean = app_all["bureau_amt_credit_debt_sum"].clip(lower=0)

# TOTAL_DEBT_TO_INCOME: Tổng nợ hiện tại + dư nợ bureau so với thu nhập
# Phản ánh toàn bộ gánh nặng nợ thực tế của khách hàng
# Thay vì dùng bureau_amt_credit_debt_sum ta dùng bureau_debt_clean
app_all["TOTAL_DEBT_TO_INCOME"] = (
    app_all["AMT_CREDIT"] + bureau_debt_clean.fillna(0)
) / app_all[
    "AMT_INCOME_TOTAL"
]  # đề phòng không có lịch sử bureau

# BUREAU_DEBT_RATIO: Dư nợ bên ngoài/có thể hiểu là dư nợ trong bureau (lịch sử) so với thu nhập
# Khác TOTAL_DEBT_TO_INCOME ở chỗ chỉ tính nợ cũ, không tính khoản vay hiện tại
app_all["BUREAU_DEBT_RATIO"] = (
    bureau_debt_clean / app_all["AMT_INCOME_TOTAL"]
).round(4)

# BUREAU_CREDIT_UTILIZATION: Tỷ lệ đang dùng so với tổng hạn mức tại bureau
# (tổng nợ thẻ/tổng hạn mức được cấp)
# Tương tự card_utilization nhưng ở cấp độ toàn bộ lịch sử bureau
# replace(0, np.nan) tránh chia cho 0
app_all["BUREAU_CREDIT_UTILIZATION"] = bureau_debt_clean / app_all[
    "bureau_amt_credit_sum_sum"
].replace(0, np.nan)

# Kiểm tra kết quả:
print("Các feature GROUP2 vừa tạo: ")
new_group2_cols = [
    "DEBT_TO_INCOME",
    "ANNUITY_TO_INCOME",
    "CREDIT_TO_ANNUITY_MONTHS",
    "IMPLICIT_INTEREST_RATE",
    "LTV",
    "INCOME_PER_CAPITA",
    "TOTAL_ANNUITY",
    "TOTAL_DEBT_TO_INCOME",
    "BUREAU_DEBT_RATIO",
    "BUREAU_CREDIT_UTILIZATION",
]
print(app_all[new_group2_cols].describe().round(4))
print("Số mising values: ")
print(app_all[new_group2_cols].isnull().sum())

In [ ]:
print(app_all['TOTAL_DEBT_TO_INCOME'].quantile([0.99, 0.999, 0.9999]))
print(app_all['BUREAU_DEBT_RATIO'].quantile([0.99, 0.999, 0.9999]))
print(app_all['BUREAU_CREDIT_UTILIZATION'].quantile([0.99, 0.999, 0.9999]))

for col in ["TOTAL_DEBT_TO_INCOME", "BUREAU_DEBT_RATIO"]:
    cap = app_all[col].quantile(0.999)
    app_all[col] = app_all[col].clip(upper=cap)

cap = app_all["BUREAU_CREDIT_UTILIZATION"].quantile(0.9999)
app_all["BUREAU_CREDIT_UTILIZATION"] = app_all["BUREAU_CREDIT_UTILIZATION"].clip(upper=cap)

### kết quả có 1 số giá trị lạ: cột `TOTAL_DEBT_TO_INCOME`, `BUREAU_DEBT_RATIO` có giá trị âm (cân nhắc chuyển 0 vì xét về mặt rủi ro, nộp thừa tiền nghĩa là họ đang có gánh nặng nợ bằng 0)
### đối với `BUREAU_CREDIT_UTILIZATION`, ra âm có thể do việc tổng dư nợ âm (tức tổ chức đang nợ khách) cũng nên cân nhắc chuyển 0

### chỉnh trực tiếp vào code gốc, nên các giá trị âm sẽ biến mất ở các lần chạy sau

In [ ]:
# ===== NHÓM 3: HÀNH VI THANH TOÁN TỔNG HỢP (CROSS-SOURCE DPD) =====

# MAX_DPD_ALL_SOURCES: DPD nặng nhất từ tất cả nguồn
# Gộp DPD từ 4 nguồn: bureau, card, pos, installments
# Dùng np.nanmax để bỏ qua NaN (khách không có lịch sử ở nguồn nào đó)
app_all["MAX_DPD_ALL_SOURCES"] = app_all[
    ["bureau_dpd_max", "card_dpd_max", "pos_dpd_max", "install_payment_delay_max"]
].max(axis=1, skipna=True)

# TOTAL_LATE_MONTHS: Tổng số lần hoặc tổng số tháng từng quá hạn từ tất cả nguồn
# Cộng dồn từ 4 nguồn, fillna(0) vì không có lịch sử = không có quá hạn
app_all["TOTAL_LATE_MONTHS"] = (
    app_all["bureau_dpd_count_sum"].fillna(0)
    + app_all["card_dpd_ever_count"].fillna(0)
    + app_all["pos_dpd_ever_count"].fillna(0)
    + app_all["install_late_count"].fillna(0)
)

# HAS_SEVERE_OVERDUE: Flag từng quá hạn nghiêm trọng (binary 0/1)
# Điều kiện: bureau từng quá hạn nặng (status >= 3) HOẶC pos từng ở trạng thái High_risk
# fillna(0) vì không có lịch sử = chưa từng quá hạn nghiêm trọng
app_all["HAS_SEVERE_OVERDUE"] = (
    (app_all["bureau_dpd_severe_ever_rate"].fillna(0) > 0)
    | (app_all["pos_high_risk_ever"].fillna(0) == 1)
).astype(int)

# DPD_RECENCY_SEVERITY: Kết hợp mức độ nặng và độ gần đây của quá hạn
# Công thức: DPD_max / (|tháng kể từ lần quá hạn gần nhất| + 1)
# Càng cao = quá hạn càng nặng VÀ càng gần đây = rủi ro cao nhất
# +1 để tránh chia cho 0 khi quá hạn xảy ra ngay tháng hiện tại
# kết quả càng lớn thì càng rủi ro
app_all["DPD_RECENCY_SEVERITY"] = app_all["bureau_dpd_max"] / (
    app_all["bureau_months_since_last_dpd_min"].abs() + 1
)

# INSTALL_COMPLETION_RATE — 2 phiên bản:

# Phiên bản A: Để NaN khi không có lịch sử → LightGBM/XGBoost tự xử lý
# Ý nghĩa: không có lịch sử = không rõ = để model tự quyết định
app_all["INSTALL_COMPLETION_RATE_NAN"] = (
    (app_all["install_count"] - app_all["install_late_count"])
    / app_all["install_count"].replace(0, np.nan)
    # install_count = 0 hoặc NaN → kết quả NaN
)

# Phiên bản B: Đặt = 0 khi không có lịch sử → coi như rủi ro
# Ý nghĩa: không có thông tin tốt = không chứng minh được = rủi ro
app_all["INSTALL_COMPLETION_RATE_ZERO"] = (
    (app_all["install_count"] - app_all["install_late_count"])
    / app_all["install_count"].replace(0, np.nan)
).fillna(0)

# INSTALLMENT_PAYMENT_GAP: Chênh lệch giữa thực trả và phải trả
# Âm = thường xuyên trả thiếu → rủi ro
# Dương = thường xuyên trả dư → tích cực
# = 0 → trả đúng số tiền quy định
app_all["INSTALLMENT_PAYMENT_GAP"] = app_all["install_payment_ratio_mean"] - 1.0

# Kiểm tra kết quả
new_group3_cols = [
    "MAX_DPD_ALL_SOURCES",
    "TOTAL_LATE_MONTHS",
    "HAS_SEVERE_OVERDUE",
    "DPD_RECENCY_SEVERITY",
    "INSTALL_COMPLETION_RATE_NAN",
    "INSTALL_COMPLETION_RATE_ZERO",
    "INSTALLMENT_PAYMENT_GAP",
]
print("Các feauture GROUP3 vừa tạo: ")
print(app_all[new_group3_cols].describe().round(4))
print("\nSố missing GROUP3: ")
print(app_all[new_group3_cols].isnull().sum())

In [ ]:
print(app_all['MAX_DPD_ALL_SOURCES'].quantile([0.99, 0.999, 0.9999]))
print(app_all['TOTAL_LATE_MONTHS'].quantile([0.99, 0.999, 0.9999]))
print(app_all['HAS_SEVERE_OVERDUE'].quantile([0.99, 0.999, 0.9999]))
print(app_all['DPD_RECENCY_SEVERITY'].quantile([0.99, 0.999, 0.9999]))
print(app_all['INSTALL_COMPLETION_RATE_NAN'].quantile([0.99, 0.999, 0.9999]))
print(app_all['INSTALL_COMPLETION_RATE_ZERO'].quantile([0.99, 0.999, 0.9999]))
print(app_all['INSTALLMENT_PAYMENT_GAP'].quantile([0.99, 0.999, 0.9999]))

In [ ]:
cap = app_all['MAX_DPD_ALL_SOURCES'].quantile(0.99)
app_all['MAX_DPD_ALL_SOURCES'] = app_all['MAX_DPD_ALL_SOURCES'].clip(upper=cap)

In [ ]:
# ===== NHÓM 4: SO SÁNH VỚI LỊCH SỬ VAY (CROSS-TABLE) =====

# CURRENT_VS_PREV_CREDIT: Khoản vay hiện tại lớn hơn khoản vay trước bao nhiêu lần
# > 1 = đang vay nhiều hơn lần trước → có thể đang gặp khó khăn tài chính hơn
# replace(0, np.nan) và fillna(np.nan) để tránh chia cho 0 hoặc NaN lan truyền
app_all["CURRENT_VS_PREV_CREDIT"] = app_all["AMT_CREDIT"] / app_all[
    "pre_amt_credit_mean"
].replace(0, np.nan)

# PRE_CREDIT_GAP: Chênh lệch giữa số tiền được duyệt và số tiền xin vay (trung bình)
# Âm = thường được duyệt ít hơn xin → tổ chức tín dụng không tin tưởng hoàn toàn
# Dương = được duyệt nhiều hơn xin → hiếm, có thể do phí/bảo hiểm
app_all["PRE_CREDIT_GAP"] = (
    app_all["pre_amt_credit_mean"] - app_all["pre_amt_application_mean"]
)

# PRE_APPROVAL_RATE: Tỷ lệ được duyệt trong số đơn đã có quyết định
# Loại trừ Canceled và Unused offer (chưa có quyết định rõ ràng)
# +1e-9 tránh chia cho 0 khi cả approved và refused đều = 0
app_all["PRE_APPROVAL_RATE"] = app_all["pre_approved_rate"] / (
    app_all["pre_approved_rate"] + app_all["pre_refused_rate"]
).replace(0, np.nan)

# PRE_CREDIT_TO_GOODS: Tỷ lệ số tiền vay / giá trị hàng hóa ở lần vay trước
# Tương tự LTV nhưng từ lịch sử — nếu luôn vay > giá trị hàng → hành vi rủi ro nhất quán
app_all["PRE_CREDIT_TO_GOODS"] = app_all["pre_amt_credit_mean"] / app_all[
    "pre_amt_goodsprice_mean"
].replace(0, np.nan)

# Kiểm tra kết quả
new_group4_cols = [
    "CURRENT_VS_PREV_CREDIT",
    "PRE_CREDIT_GAP",
    "PRE_APPROVAL_RATE",
    "PRE_CREDIT_TO_GOODS",
]
print("Các feature GROUP4 vừa tạo:")
print(app_all[new_group4_cols].describe().round(4))
print("\nSố NaN:")
print(app_all[new_group4_cols].isnull().sum())

In [ ]:
# Kiểm tra tiếp các outlier
print(app_all['CURRENT_VS_PREV_CREDIT'].quantile([0.99, 0.999, 0.9999]))
print(app_all['PRE_CREDIT_TO_GOODS'].quantile([0.99, 0.999, 0.9999]))

# Bỏ PRE_CREDIT_GAP vì 2 cột gốc gần như đồng nhất
app_all = app_all.drop(columns=['PRE_CREDIT_GAP'])

# Clip CURRENT_VS_PREV_CREDIT theo 99.9%
cap = app_all['CURRENT_VS_PREV_CREDIT'].quantile(0.999)
app_all['CURRENT_VS_PREV_CREDIT'] = app_all['CURRENT_VS_PREV_CREDIT'].clip(upper=cap)

In [ ]:
# ===== NHÓM 5: ĐỘ ỔN ĐỊNH NHÂN THÂN =====

# EMPLOYMENT_LIFE_RATIO: Tỷ lệ thời gian đi làm so với tuổi đời
# Càng cao = đã đi làm được nhiều % cuộc đời = ổn định hơn
# DAYS_EMPLOYED đã được xử lý NaN cho nhóm không đi làm (DAYS_EMPLOYED_ANOM)
# nên NaN ở đây có nghĩa là "không đi làm chính thức" → giữ NaN, không fillna
app_all["EMPLOYMENT_LIFE_RATIO"] = app_all["DAYS_EMPLOYED"] / app_all["DAYS_BIRTH"]

# RESIDENCE_STABILITY: Tỷ lệ thời gian đăng ký giấy tờ so với tuổi
# Càng cao = ít đổi chỗ ở = ổn định hơn
app_all["RESIDENCE_STABILITY"] = app_all["DAYS_REGISTRATION"] / app_all["DAYS_BIRTH"]

# PHONE_STABILITY: Tỷ lệ thời gian dùng SĐT so với tuổi
# Càng cao = dùng SĐT lâu = ổn định hơn
app_all["PHONE_STABILITY"] = app_all["DAYS_LAST_PHONE_CHANGE"] / app_all["DAYS_BIRTH"]

# ID_STABILITY: Tỷ lệ thời gian kể từ khi cấp CMND so với tuổi
# CMND lâu chưa đổi = thông tin nhân thân ổn định
app_all["ID_STABILITY"] = app_all["DAYS_ID_PUBLISH"] / app_all["DAYS_BIRTH"]

# ADDRESS_MISMATCH_SCORE: Tổng điểm sai lệch địa chỉ
# Cộng dồn 3 flag địa chỉ không khớp (cấp thành phố)
# Càng cao = địa chỉ càng không nhất quán = ít ổn định hơn
# Dùng fillna(0) vì các cột FLAG không có NaN trong dataset này
app_all["ADDRESS_MISMATCH_SCORE"] = (
    app_all["REG_REGION_NOT_LIVE_REGION"].fillna(0)
    + app_all["REG_CITY_NOT_LIVE_CITY"].fillna(0)
    + app_all["REG_CITY_NOT_WORK_CITY"].fillna(0)
)

# Kiểm tra kết quả
new_group5_cols = [
    "EMPLOYMENT_LIFE_RATIO",
    "RESIDENCE_STABILITY",
    "PHONE_STABILITY",
    "ID_STABILITY",
    "ADDRESS_MISMATCH_SCORE",
]
print("Các feature GROUP5 vừa tạo:")
print(app_all[new_group5_cols].describe().round(4))
print("\nSố NaN:")
print(app_all[new_group5_cols].isnull().sum())

In [ ]:
# ===== NHÓM 6: THẺ TÍN DỤNG (CREDIT CARD BEHAVIOR) =====

# CARD_PAYMENT_TO_BALANCE: Tỷ lệ tiền thanh toán / số dư thẻ
# > 1: trả nhiều hơn số dư hiện tại → tích cực
# = 1: trả đúng số dư → tốt
# < 1: chỉ trả một phần → có thể đang tích lũy nợ
# replace(0, np.nan): tránh chia cho 0 khi số dư = 0
app_all["CARD_PAYMENT_TO_BALANCE"] = app_all["card_amt_payment_mean"] / app_all[
    "card_amt_balance_mean"
].replace(0, np.nan)

# CARD_DRAWINGS_TO_LIMIT: Tỷ lệ số tiền rút / hạn mức thẻ
# Càng cao = đang dùng gần hết hạn mức để rút tiền mặt
# Rút tiền mặt nhiều thường là dấu hiệu thiếu thanh khoản → rủi ro cao
# replace(0, np.nan): tránh chia cho 0 khi hạn mức = 0
app_all["CARD_DRAWINGS_TO_LIMIT"] = app_all["card_amt_drawings_mean"] / app_all[
    "card_amt_credit_limit_mean"
].replace(0, np.nan)

# Kiểm tra kết quả
new_group6_cols = ["CARD_PAYMENT_TO_BALANCE", "CARD_DRAWINGS_TO_LIMIT"]
print("Các feature GROUP6 vừa tạo:")
print(app_all[new_group6_cols].describe().round(4))
print("\nSố NaN:")
print(app_all[new_group6_cols].isnull().sum())

In [ ]:
print(app_all["CARD_DRAWINGS_TO_LIMIT"].quantile([0.99, 0.999, 0.9999]))
# Clip CARD_DRAWINGS_TO_LIMIT theo 99.9%
cap = app_all["CARD_DRAWINGS_TO_LIMIT"].quantile(0.999)
app_all["CARD_DRAWINGS_TO_LIMIT"] = app_all["CARD_DRAWINGS_TO_LIMIT"].clip(upper=cap)
app_all = app_all.drop(columns=["CARD_PAYMENT_TO_BALANCE"])

In [ ]:
# ===== NHÓM 7: VÒNG TRÒN XÃ HỘI =====

# DEF_RATE_30: Tỷ lệ vỡ nợ trong vòng tròn xã hội (30 ngày)
# = số người vỡ nợ / số người được quan sát trong 30 ngày
# +1e-9 tránh chia cho 0 khi không có ai được quan sát
# Càng cao = bạn bè/người thân vỡ nợ nhiều → môi trường tài chính xấu
app_all["DEF_RATE_30"] = app_all["DEF_30_CNT_SOCIAL_CIRCLE"] / (
    app_all["OBS_30_CNT_SOCIAL_CIRCLE"] + 1e-9
)

# DEF_RATE_60: Tỷ lệ vỡ nợ trong vòng tròn xã hội (60 ngày)
# Tương tự DEF_RATE_30 nhưng window dài hơn → ổn định hơn về mặt thống kê
app_all["DEF_RATE_60"] = app_all["DEF_60_CNT_SOCIAL_CIRCLE"] / (
    app_all["OBS_60_CNT_SOCIAL_CIRCLE"] + 1e-9
)

# SOCIAL_TREND_WORSE: Xu hướng xấu đi của vòng tròn xã hội
# = DEF_RATE_60 - DEF_RATE_30
# Dương = tỷ lệ vỡ nợ 60 ngày cao hơn 30 ngày → bạn bè gần đây vỡ nợ nhiều hơn
# Âm = tỷ lệ vỡ nợ đang cải thiện
# = 0 = không thay đổi
app_all["SOCIAL_TREND_WORSE"] = app_all["DEF_RATE_60"] - app_all["DEF_RATE_30"]

# Kiểm tra kết quả
new_group7_cols = ["DEF_RATE_30", "DEF_RATE_60", "SOCIAL_TREND_WORSE"]
print("Các feature GROUP7 vừa tạo:")
print(app_all[new_group7_cols].describe().round(4))
print("\nSố NaN:")
print(app_all[new_group7_cols].isnull().sum())

In [ ]:
app_all = app_all.drop(columns=['SOCIAL_TREND_WORSE'])

In [ ]:
# Nhóm 8 — Cờ báo động đỏ (Red Flags)

# RECENT_INQUIRY_SPIKE: Phát hiện cơn khát tín dụng bất thường
# Tỷ lệ tra cứu thông tin tín dụng trong tuần gần nhất so với mức trung bình năm.
# Dùng np.select để xử lý phân tách logic điều kiện mà không làm phát sinh lỗi chia cho 0.
# Giả định tên cột gốc của bạn là AMT_REQ_CREDIT_BUREAU_YEAR và AMT_REQ_CREDIT_BUREAU_WEEK
year_inq = app_all["AMT_REQ_CREDIT_BUREAU_YEAR"]
week_inq = app_all["AMT_REQ_CREDIT_BUREAU_WEEK"]

cond_normal = (year_inq == 0) & (week_inq == 0)  # Không có tra cứu nào
cond_spike = (year_inq == 0) & (
    week_inq > 0
)  # Báo động đỏ: Bỗng dưng phát sinh tra cứu tuần này mà lịch sử năm trống

# Công thức tính cho các trường hợp còn lại (lúc này YEAR chắc chắn > 0 hoặc NaN)
inquiry_ratio = week_inq / (year_inq / 52)

app_all["RECENT_INQUIRY_SPIKE"] = np.select(
    condlist=[cond_normal, cond_spike], choicelist=[1.0, 2.0], default=inquiry_ratio
).round(4)


# TOTAL_MISSING_FLAGS: Tổng số lượng thông tin bị khuyết (Missing Values)
# Ý nghĩa kinh tế: Khách hàng cố tình giấu thông tin hoặc hồ sơ quá "trắng" -> Càng nhiều missing càng rủi ro
# Chú ý: Đảm bảo các cột _MISSING dưới đây đã được bạn định nghĩa ở dạng 0 hoặc 1 ở các bước tiền xử lý trước
app_all["TOTAL_MISSING_FLAGS"] = (
    app_all["EXT_SOURCE_1_MISSING"]
    + app_all["EXT_SOURCE_3_MISSING"]
    + app_all["BDS_INFO_MISSING"]
    + app_all["AMT_REQ_MISSING"]
    + app_all["OWN_CAR_AGE_MISSING"]
    + app_all["card_missing"]
)


# IS_OFFICIAL_EMPLOYED: Xác định nhóm có công việc thuộc diện chính thức
# Thu nhập chính thức ổn định hơn, có hợp đồng lao động rõ ràng -> Rủi ro nhảy việc hoặc bùng nợ thấp hơn
app_all["IS_OFFICIAL_EMPLOYED"] = (
    app_all["NAME_INCOME_TYPE"].isin(["Working", "State servant"]).astype(int)
)

# Kiểm tra kết quả:
print("Các feature GROUP8 vừa tạo: ")
new_group8_cols = [
    "RECENT_INQUIRY_SPIKE",
    "TOTAL_MISSING_FLAGS",
    "IS_OFFICIAL_EMPLOYED",
]
print(app_all[new_group8_cols].describe().round(4))
print("\nSố missing values: ")
print(app_all[new_group8_cols].isnull().sum())

In [ ]:
print(app_all["RECENT_INQUIRY_SPIKE"].quantile([0.99, 0.999, 0.9999]))

In [ ]:
# Clip RECENT_INQUIRY_SPIKE theo 99.99%
cap = app_all["RECENT_INQUIRY_SPIKE"].quantile(0.9999)
app_all["RECENT_INQUIRY_SPIKE"] = app_all["RECENT_INQUIRY_SPIKE"].clip(upper=cap)

In [ ]:
missing_flag_cols = [
    'EXT_SOURCE_1_MISSING', 'EXT_SOURCE_3_MISSING',
    'BDS_INFO_MISSING', 'AMT_REQ_MISSING',
    'OWN_CAR_AGE_MISSING', 'card_missing'
]
print(app_all[missing_flag_cols].isnull().sum())

In [ ]:
# Chuyển các cột dạng chữ (object) sang 'category' để LightGBM tự học trực tiếp
# Không cần phải One-Hot Encoding hay Label Encoding thủ công nữa.
for col in app_all.columns:
    if app_all[col].dtype == "object":
        app_all[col] = app_all[col].astype("category")

# Tách ngược lại train/test
app_train_fe = app_all.iloc[:n_train].copy()
app_test_fe = app_all.iloc[n_train:].copy()


# Gắn TARGET lại vào train
app_train_fe["TARGET"] = target.values

print(f"app_train_fe: {app_train_fe.shape}")
print(f"app_test_fe: {app_test_fe.shape}")
print(
    f"TARGET distribution:\n{app_train_fe['TARGET'].value_counts(normalize=True).round(4)}"
)

In [ ]:
app_train_fe.to_parquet("../Data/app_train_fe.parquet")
app_test_fe.to_parquet("../Data/app_test_fe.parquet")
print("Đã lưu 2 file app_train_fe và app_test_fe thành Parquet để lưu trữ kiểu dữ liệu category")

# ==============================================================================
# GIẢI THÍCH OPTUNA
# ==============================================================================
# Optuna là framework tự động tìm hyperparameter tốt nhất theo cơ chế:
# 1. Thử 1 bộ tham số (gọi là 1 "trial")
# 2. Đánh giá kết quả (AUC)
# 3. Học từ kết quả đó để quyết định thử bộ tham số nào tiếp theo
# 4. Lặp lại n_trials lần
# Thuật toán sử dụng: TPE (Tree-structured Parzen Estimator)
# — Ước tính xác suất bộ tham số nào cho kết quả tốt dựa trên lịch sử các trial trước

# ==============================================================================
# BƯỚC 1: ĐỊNH NGHĨA "OBJECTIVE FUNCTION"
# ==============================================================================
# Đây là hàm Optuna sẽ gọi lặp đi lặp lại
# Mỗi lần gọi = 1 trial với 1 bộ hyperparameter khác nhau
# Optuna sẽ cố maximize (hoặc minimize) giá trị return của hàm này


def objective(trial):
    # trial.suggest_*: Optuna đề xuất giá trị cho từng tham số
    # Mỗi lần gọi objective(), Optuna chọn giá trị khác nhau dựa trên lịch sử

    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "random_state": 42,
        "n_jobs": -1,
        # Số lá mỗi cây — càng lớn = model càng phức tạp, dễ overfit
        # suggest_int: chọn số nguyên trong khoảng [20, 300]
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        # Learning rate — càng nhỏ = học chậm nhưng ổn định hơn
        # suggest_float với log=True: tìm kiếm theo thang log (0.01, 0.1 quan trọng như nhau)
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        # Số feature dùng mỗi cây (colsample) — giảm overfitting
        # suggest_float: chọn số thực trong khoảng [0.4, 1.0]
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        # Số dòng dùng mỗi cây (subsample) — giảm overfitting
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        # Số dòng tối thiểu để tạo 1 lá — càng lớn = model càng đơn giản
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        # L1 regularization — phạt các feature ít quan trọng về 0
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        # L2 regularization — giảm magnitude của tất cả feature
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        # Số feature dùng cho mỗi split (khác colsample_bytree ở cấp độ split)
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        # Số dòng dùng cho mỗi cây (tương tự subsample nhưng tên khác trong LightGBM)
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        # Tần suất áp dụng bagging — 1 = áp dụng mỗi iteration
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        # Số cây tối đa — early stopping sẽ dừng sớm hơn nếu không cải thiện
        "n_estimators": 2000,
    }

    # Dùng 3-fold thay vì 5-fold để tiết kiệm thời gian tune
    # Sau khi tìm được tham số tốt, sẽ train lại với 5-fold để đánh giá chính xác
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in skf.split(X_train[features], y_train):
        X_tr = X_train[features].iloc[train_idx]
        X_val = X_train[features].iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
            categorical_feature=cat_cols,
        )

        val_preds = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, val_preds))

    # Trả về AUC trung bình 3 fold — Optuna sẽ cố maximize giá trị này
    return np.mean(auc_scores)


# ==============================================================================
# BƯỚC 2: TẠO VÀ CHẠY STUDY
# ==============================================================================

# Study = 1 phiên tìm kiếm hyperparameter
# direction='maximize': Optuna sẽ tìm bộ tham số cho AUC cao nhất
# sampler=TPESampler: dùng thuật toán TPE để tìm kiếm thông minh
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))

# Tắt log mặc định của Optuna cho gọn
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Bắt đầu Optuna tuning...")
print("Mỗi trial = 1 lần thử bộ hyperparameter với 3-fold CV")
print("=" * 50)

# optimize(): chạy n_trials lần gọi objective()
# n_trials=50: thử 50 bộ tham số khác nhau
# Tăng n_trials = kết quả tốt hơn nhưng tốn thời gian hơn
# Với dataset lớn như này, 20-100 trials là hợp lý, chọn 20 cho nhanh
study.optimize(
    objective, n_trials=20, show_progress_bar=True  # Hiển thị thanh tiến trình
)

# ==============================================================================
# BƯỚC 3: XEM KẾT QUẢ
# ==============================================================================

print(f"\nBest AUC: {study.best_value:.4f}")
print(f"Best GINI: {study.best_value * 2 - 1:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Vẽ biểu đồ lịch sử optimization
# Thấy được AUC cải thiện qua từng trial như thế nào
optuna.visualization.plot_optimization_history(study).show()

# Vẽ biểu đồ tầm quan trọng của từng hyperparameter
# Tham số nào ảnh hưởng nhiều nhất đến AUC
optuna.visualization.plot_param_importances(study).show()